# S9 - Observabilidad con Grafana y costos

**Actividad:** instrumentar el pipeline de S8 (`atmos-eventos`) con campos de observabilidad calculados en Spark (`isValid`, `processedAt`, `latencyMs`), conectar Kafka a Prometheus, armar un tablero real en Grafana, calcular latencia agregada por ventana de tiempo, y proponer alertas y una estimacion de costos basadas en datos reales.

## Objetivo

1. Reanudar Kafka y el productor/consumidor de S7, y levantar Prometheus + Grafana unidos a la misma red.
2. Crear la `SparkSession` con el conector de Kafka (el mismo arranque de S8).
3. Leer `atmos-eventos` y calcular tres columnas nuevas: `isValid`, `processedAt`, `latencyMs`.
4. Ver los eventos en consola, con sus campos de observabilidad.
5. Revisar throughput y lag con `lastProgress`.
6. Persistir la evidencia observable en Parquet, en paralelo a la consulta de consola.

## Entorno

Este notebook asume que ya completaste 3.1-3.3 de la guia (Kafka, uso-atmos con el productor corriendo en vivo, y obs/ con Prometheus+Grafana levantado) y que estas corriendo esta celda desde JupyterLab, osea con `pyspark` ya unido a `lambda26-kafka-net` (`docker compose -f compose.yml -f compose.kafka.yml up -d`, ver `pyspark/README.md`). Si algo de eso falta, el onboarding completo vive en la guia y en el README de cada carpeta (`kafka/`, `uso-atmos/`, `obs/`), no aqui.

## 3.4 Crear el notebook y la SparkSession, con el conector de Kafka

**Producto del paso:** notebook `09_observabilidad_pipeline_grafana_costos.ipynb` con una SparkSession capaz de leer y escribir Kafka - el mismo arranque que ya hiciste en S8.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion9-observabilidad")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0b0e8f7d-ac8e-40bf-889c-a76df527ed76;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

## 3.5 Leer el topic y calcular los campos de observabilidad

**Producto del paso:** un DataFrame con los campos de atmos-eventos ya tipados, mas tres columnas nuevas que no existian en S8: isValid, processedAt y latencyMs.

In [2]:
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temperatura", DoubleType())
    .add("humedad", DoubleType())
    .add("presion", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "atmos-eventos")
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.select(
        col("topic"), col("partition"), col("offset"),
        col("timestamp").alias("kafkaTimestamp"),
        col("value").cast("string").alias("json_str"),
    )
    .select(
        "topic", "partition", "offset", "kafkaTimestamp",
        from_json(col("json_str"), esquema_evento).alias("e"),
    )
    .select("topic", "partition", "offset", "kafkaTimestamp", "e.*")
)

observables = (
    eventos
    .withColumn(
        "isValid",
        col("tipoEvento").isNotNull()
        & col("sensorId").isNotNull()
        & col("temperatura").isNotNull()
        & col("timestamp").isNotNull(),
    )
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
)
observables.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



kafkaTimestamp (metadata de Kafka: cuando el broker recibio el mensaje) es distinto de timestamp (un campo del propio evento: cuando el sensor midio, igual que en S8). processedAt es el momento en que esta celda de Spark proceso el evento - y latencyMs es la diferencia entre ambos: cuanto tardo un evento en viajar de producer_sensores.py, por Kafka, hasta este punto del pipeline. isValid reusa la misma idea de contrato minimo que ya viste en S7 (consumer_sensores.py), pero calculada en Spark en vez de en Python simple.

## 3.6 Ver los eventos en consola, con sus campos de observabilidad

**Producto del paso:** la salida de consola de S8, ahora con isValid, processedAt y latencyMs visibles fila por fila.

In [3]:
# Momento 1: arrancar la consulta
consulta = (
    observables.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

-------------------------------------------
Batch: 0
-------------------------------------------
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+
|topic|partition|offset|kafkaTimestamp|tipoEvento|sensorId|temperatura|humedad|presion|origen|timestamp|isValid|processedAt|latencyMs|
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2622  |2026-10-09 02:11:30.174|sensor.lectura|esp32-patio      |19.0       |84.0   |995.3  |uso-atmos|1791511890174|true   |1791511892133|1959     |
|atmos-eventos|0        |2623  |2026-10-09 02:11:30.177|sensor.lectura|esp32-invernadero|19.8       |52.2   |1018.2 |uso-atmos|1791511890177|true   |1791511892133|1956     |
|atmos-eventos|0        |2624  |2

Dejala corriendo - no la detengas todavia. Vas a usarla en 3.7 para revisar lastProgress, y en 3.8 vas a arrancar una segunda consulta que persiste en Parquet al mismo tiempo. Las vas a detener juntas al final de 3.8.

## 3.7 Revisar throughput y lag con lastProgress

**Producto del paso:** una lectura real de consulta.lastProgress, con el throughput del ultimo micro-lote y el offset de cada particion de Kafka.

In [4]:
import json

progreso = consulta.lastProgress

if progreso is None:
    print("Todavia no hay micro-lote procesado. Espera unos segundos y corre esta celda de nuevo.")
else:
    fuente = progreso["sources"][0]
    metricas = fuente.get("metrics", {})
    estado = "idle" if progreso["numInputRows"] == 0 else "processed"

    log = {
        "service": "spark-streaming",
        "component": "observabilidad",
        "topic": "atmos-eventos",
        "batchId": progreso["batchId"],
        "numInputRows": progreso["numInputRows"],
        "inputRowsPerSecond": progreso["inputRowsPerSecond"],
        "processedRowsPerSecond": progreso["processedRowsPerSecond"],
        "avgOffsetsBehindLatest": metricas.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": metricas.get("maxOffsetsBehindLatest"),
        "status": estado,
    }
    print(json.dumps(log, indent=2))

{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "atmos-eventos",
  "batchId": 6,
  "numInputRows": 6,
  "inputRowsPerSecond": 1.2,
  "processedRowsPerSecond": 6.8,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "status": "processed"
}


-------------------------------------------
Batch: 7
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2649  |2026-10-09 02:11:57.335|sensor.lectura|esp32-patio      |18.9       |83.3   |995.4  |uso-atmos|1791511917334|true   |1791511920014|2680     |
|atmos-eventos|0        |2650  |2026-10-09 02:11:57.339|sensor.lectura|esp32-invernadero|19.4       |47.8   |1017.7 |uso-atmos|1791511917339|true   |1791511920014|2675     |
|atmos-eventos|0        |2651  |2

-------------------------------------------
Batch: 11
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2667  |2026-10-09 02:12:15.433|sensor.lectura|esp32-patio      |18.8       |81.7   |995.5  |uso-atmos|1791511935433|true   |1791511940034|4601     |
|atmos-eventos|0        |2668  |2026-10-09 02:12:15.437|sensor.lectura|esp32-invernadero|19.6       |45.4   |1018.6 |uso-atmos|1791511935437|true   |1791511940034|4597     |
|atmos-eventos|0        |2669  |

numInputRows es cuantos eventos trajo el ultimo micro-lote; inputRowsPerSecond/processedRowsPerSecond son la velocidad de entrada y de procesamiento - si processedRowsPerSecond es mayor que inputRowsPerSecond, Spark esta procesando mas rapido de lo que llegan los datos, que es la situacion sana. avgOffsetsBehindLatest y maxOffsetsBehindLatest son el lag visto desde el lado de Spark: cuanto offset de Kafka le falta por leer a esta consulta en este momento. Si numInputRows sale 0, no es un error - significa que ese micro-lote en particular no encontro eventos nuevos (status: "idle").

Evidencia alternativa - Spark UI: abre http://localhost:4040 y entra a Structured Streaming -> Streaming Query Statistics. Ahi ves los mismos numeros (Input Rate, Process Rate, Batch Duration) de forma grafica, sin tener que leer lastProgress a mano.

## 3.8 Persistir la evidencia observable en Parquet

**Producto del paso:** los eventos con sus campos de observabilidad aterrizando en disco, al mismo tiempo que la consulta de consola de 3.6 sigue corriendo.

In [5]:
# Momento 1: arrancar el stream a Parquet (la consulta de 3.6 sigue corriendo en paralelo)
ARTIFACTS = "./artifacts/atmos_eventos_observabilidad"
CHECKPOINT_PARQUET = "./artifacts/chk_observabilidad"

consulta2 = (
    observables.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ARTIFACTS)
    .option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime="5 seconds")
    .start()
)

-------------------------------------------
Batch: 23
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2727  |2026-10-09 02:13:15.77 |sensor.lectura|esp32-patio      |19.1       |78.6   |995.4  |uso-atmos|1791511995770|true   |1791512000010|4240     |
|atmos-eventos|0        |2728  |2026-10-09 02:13:15.773|sensor.lectura|esp32-invernadero|20.7       |45.3   |1018.5 |uso-atmos|1791511995773|true   |1791512000010|4237     |
|atmos-eventos|0        |2729  |

-------------------------------------------
Batch: 24
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2733  |2026-10-09 02:13:21.796|sensor.lectura|esp32-patio      |18.9       |78.8   |995.6  |uso-atmos|1791512001795|true   |1791512005010|3215     |
|atmos-eventos|0        |2734  |2026-10-09 02:13:21.798|sensor.lectura|esp32-invernadero|20.8       |43.7   |1017.7 |uso-atmos|1791512001798|true   |1791512005010|3212     |
|atmos-eventos|0        |2735  |

-------------------------------------------
Batch: 25
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2739  |2026-10-09 02:13:27.839|sensor.lectura|esp32-patio      |18.6       |78.7   |995.8  |uso-atmos|1791512007839|true   |1791512010018|2179     |
|atmos-eventos|0        |2740  |2026-10-09 02:13:27.842|sensor.lectura|esp32-invernadero|20.9       |45.0   |1017.1 |uso-atmos|1791512007842|true   |1791512010018|2176     |
|atmos-eventos|0        |2741  |

-------------------------------------------
Batch: 26
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2742  |2026-10-09 02:13:30.862|sensor.lectura|esp32-patio      |18.8       |78.0   |996.2  |uso-atmos|1791512010861|true   |1791512015037|4176     |
|atmos-eventos|0        |2743  |2026-10-09 02:13:30.879|sensor.lectura|esp32-invernadero|21.1       |45.4   |1016.9 |uso-atmos|1791512010879|true   |1791512015037|4158     |
|atmos-eventos|0        |2744  |

-------------------------------------------
Batch: 27
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2748  |2026-10-09 02:13:36.98 |sensor.lectura|esp32-patio      |18.8       |78.5   |996.5  |uso-atmos|1791512016980|true   |1791512020070|3090     |
|atmos-eventos|0        |2749  |2026-10-09 02:13:36.984|sensor.lectura|esp32-invernadero|21.3       |45.6   |1016.8 |uso-atmos|1791512016984|true   |1791512020070|3086     |
|atmos-eventos|0        |2750  |

-------------------------------------------
Batch: 28
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2753  |2026-10-09 02:13:40.021|sensor.lectura|esp32-laboratorio|12.5       |70.9   |1019.5 |uso-atmos|1791512020021|true   |1791512025008|4987     |
|atmos-eventos|0        |2754  |2026-10-09 02:13:43.07 |sensor.lectura|esp32-patio      |18.7       |76.7   |996.3  |uso-atmos|1791512023070|true   |1791512025008|1938     |
|atmos-eventos|0        |2755  |

-------------------------------------------
Batch: 30
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2763  |2026-10-09 02:13:52.137|sensor.lectura|esp32-patio      |19.1       |76.1   |996.1  |uso-atmos|1791512032136|true   |1791512035015|2879     |
|atmos-eventos|0        |2764  |2026-10-09 02:13:52.142|sensor.lectura|esp32-invernadero|20.9       |47.2   |1016.2 |uso-atmos|1791512032142|true   |1791512035015|2873     |
|atmos-eventos|0        |2765  |

-------------------------------------------
Batch: 31
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2766  |2026-10-09 02:13:55.152|sensor.lectura|esp32-patio      |19.1       |75.6   |996.1  |uso-atmos|1791512035152|true   |1791512040016|4864     |
|atmos-eventos|0        |2767  |2026-10-09 02:13:55.156|sensor.lectura|esp32-invernadero|20.8       |47.3   |1015.9 |uso-atmos|1791512035156|true   |1791512040016|4860     |
|atmos-eventos|0        |2768  |

-------------------------------------------
Batch: 33
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2778  |2026-10-09 02:14:07.203|sensor.lectura|esp32-patio      |18.9       |75.2   |996.7  |uso-atmos|1791512047203|true   |1791512050015|2812     |
|atmos-eventos|0        |2779  |2026-10-09 02:14:07.206|sensor.lectura|esp32-invernadero|20.0       |47.6   |1015.1 |uso-atmos|1791512047206|true   |1791512050015|2809     |
|atmos-eventos|0        |2780  |

Nota el nombre consulta2, no consulta: la consulta de 3.6 sigue viva en la variable consulta, y si esta nueva reusara ese mismo nombre, perderias la unica referencia que te permite detenerla mas tarde.

Mientras corre, puedes leer lo que ya escribio sin detenerla - este paso es opcional:

In [6]:
# Momento 2 (opcional): leer lo guardado (se puede repetir, el stream sigue corriendo)
guardado = spark.read.parquet(ARTIFACTS)
total = guardado.count()
print("filas guardadas:", total)
guardado.select("sensorId", "latencyMs", "isValid").show(10, truncate=False)

-------------------------------------------
Batch: 34
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2781  |2026-10-09 02:14:10.215|sensor.lectura|esp32-patio      |19.0       |74.8   |996.6  |uso-atmos|1791512050215|true   |1791512055015|4800     |
|atmos-eventos|0        |2782  |2026-10-09 02:14:10.218|sensor.lectura|esp32-invernadero|20.1       |48.2   |1015.5 |uso-atmos|1791512050218|true   |1791512055015|4797     |
|atmos-eventos|0        |2783  |

filas guardadas: 48
+-----------------+---------+-------+
|sensorId         |latencyMs|isValid|
+-----------------+---------+-------+
|esp32-patio      |3850     |true   |
|esp32-invernadero|3847     |true   |
|esp32-laboratorio|3844     |true   |
|esp32-patio      |839      |true   |
|esp32-invernadero|837      |true   |
|esp32-laboratorio|832      |true   |
|esp32-patio      |4580     |true   |
|esp32-invernadero|4577     |true   |
|esp32-laboratorio|4572     |true   |
|esp32-patio      |1562     |true   |
+-----------------+---------+-------+
only showing top 10 rows
-------------------------------------------
Batch: 35
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|pr

-------------------------------------------
Batch: 37
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2796  |2026-10-09 02:14:25.294|sensor.lectura|esp32-patio      |19.4       |75.8   |996.9  |uso-atmos|1791512065294|true   |1791512070011|4717     |
|atmos-eventos|0        |2797  |2026-10-09 02:14:25.297|sensor.lectura|esp32-invernadero|20.6       |45.9   |1015.1 |uso-atmos|1791512065297|true   |1791512070011|4714     |
|atmos-eventos|0        |2798  |

-------------------------------------------
Batch: 39
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2808  |2026-10-09 02:14:37.352|sensor.lectura|esp32-patio      |19.2       |77.0   |996.7  |uso-atmos|1791512077352|true   |1791512080011|2659     |
|atmos-eventos|0        |2809  |2026-10-09 02:14:37.356|sensor.lectura|esp32-invernadero|20.4       |45.7   |1015.5 |uso-atmos|1791512077356|true   |1791512080011|2655     |
|atmos-eventos|0        |2810  |

-------------------------------------------
Batch: 40
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |2811  |2026-10-09 02:14:40.364|sensor.lectura|esp32-patio      |19.1       |77.7   |996.6  |uso-atmos|1791512080363|true   |1791512085015|4652     |
|atmos-eventos|0        |2812  |2026-10-09 02:14:40.367|sensor.lectura|esp32-invernadero|20.1       |46.5   |1015.5 |uso-atmos|1791512080367|true   |1791512085015|4648     |
|atmos-eventos|0        |2813  |

Cuando ya viste suficiente, deten las dos consultas juntas - la de consola de 3.6 y la de Parquet de este paso:

In [7]:
# Momento 3: detener ambas consultas (la de consola de 3.6 y la de Parquet de 3.8)
consulta.stop()
consulta2.stop()

Error frecuente: si el log muestra lineas ERROR WriteToDataSourceV2Exec, TaskKilledException o Aborting task al llamar .stop(), es el mismo ruido benigno que ya viste en S8 - Spark cancelando una tarea a mitad de un micro-lote, no una falla real.